# SynthTextEval generic evaluation demo for ICL-PILOT

This is an ICL-PILOT adaptation of the upstream [SynthTextEval demo](https://github.com/kr-ramesh/synthtexteval/blob/main/demo.ipynb), retrieved from its main branch on 2026-08-26. It retains the upstream examples for reference, but the runnable ICL-PILOT path configuration and generic text-distribution sections use the traceable all-age story-text export below.

Scope: use this notebook only for SynthTextEval's generic descriptive and automated text-quality outputs. Do not run the upstream classification, fairness, or privacy examples for the project. The project-specific DLD/SLI evaluation remains separate and is not an output of this notebook.

In [ ]:
# Colab/local repository paths. The Colab branch clones the public ICL-PILOT repo only when it is absent.
from pathlib import Path
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
COLAB_REPO_DIR = Path("/content/ICL-PILOT")
candidate_roots = [COLAB_REPO_DIR, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidate_roots if (path / "pyproject.toml").exists()), None)

if PROJECT_ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Run this notebook from the ICL-PILOT repository or open it in Colab.")
    !git clone {REPO_URL} {COLAB_REPO_DIR}
    PROJECT_ROOT = COLAB_REPO_DIR

%cd {PROJECT_ROOT}

TEXT_RECORDS_CSV = PROJECT_ROOT / "phase2/generated/all_ages/text/all_story_text_records_4_to_7.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs/synthtexteval_demo"
TEXT_COLUMN = "story_text"
SOURCE_COLUMN = "source"
GROUP_COLUMN = "group"
COHORT_COLUMN = "cohort_year"
COMPARISON_COHORTS = (4, 5, 6, 7)
COMPARISON_GROUPS = ("TD", "SLI")  # Literal corpus labels, used only to balance generic source/reference samples.
SAMPLE_SEED = 42
MAX_SAMPLES_PER_GROUP = None  # Cap each age-by-group stratum for a smaller Colab smoke test.
RUN_MAUVE = False  # Enable on a GPU runtime after the descriptive/FID/perplexity checks complete.

assert TEXT_RECORDS_CSV.exists(), f"Missing text record: {TEXT_RECORDS_CSV}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print({"project_root": str(PROJECT_ROOT), "text_records": str(TEXT_RECORDS_CSV), "results_dir": str(RESULTS_DIR)})

In [ ]:
%pip -q install "SynthTextEval==1.1.1" pyLDAvis
!python -m spacy download en_core_web_sm -q

In [ ]:
import pandas as pd

text_records = pd.read_csv(TEXT_RECORDS_CSV)
required_columns = {SOURCE_COLUMN, GROUP_COLUMN, COHORT_COLUMN, TEXT_COLUMN}
missing_columns = required_columns.difference(text_records.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

synthetic_parts, reference_parts = [], []
for cohort_year in COMPARISON_COHORTS:
    for group in COMPARISON_GROUPS:
        synthetic_group = text_records.loc[
            (text_records[SOURCE_COLUMN] == "synthetic")
            & (text_records[COHORT_COLUMN] == cohort_year)
            & (text_records[GROUP_COLUMN] == group),
            [TEXT_COLUMN],
        ].dropna()
        reference_group = text_records.loc[
            (text_records[SOURCE_COLUMN] == "real")
            & (text_records[COHORT_COLUMN] == cohort_year)
            & (text_records[GROUP_COLUMN] == group),
            [TEXT_COLUMN],
        ].dropna()
        n = min(len(synthetic_group), len(reference_group))
        if MAX_SAMPLES_PER_GROUP is not None:
            n = min(n, MAX_SAMPLES_PER_GROUP)
        assert n > 0, f"No comparable {cohort_year}-year-old {group} text records found."
        seed = SAMPLE_SEED + 100 * cohort_year + (0 if group == "TD" else 1)
        synthetic_parts.append(synthetic_group.sample(n=n, random_state=seed).reset_index(drop=True))
        reference_parts.append(reference_group.sample(n=n, random_state=seed).reset_index(drop=True))

synthetic_texts = pd.concat(synthetic_parts, ignore_index=True)[TEXT_COLUMN].tolist()
reference_texts = pd.concat(reference_parts, ignore_index=True)[TEXT_COLUMN].tolist()
comparison_df = pd.DataFrame({"source": synthetic_texts, "reference": reference_texts})
print({"synthetic_texts": len(synthetic_texts), "reference_texts": len(reference_texts), "cohorts": COMPARISON_COHORTS, "groups": COMPARISON_GROUPS})
display(comparison_df.head())

### Upstream downstream-utility example (not configured for ICL-PILOT)

Do not execute this section. ICL-PILOT's task-specific held-out TD-vs-SLI evaluation is implemented and reported separately; the paths below remain upstream TAB examples for provenance only.

In [ ]:
import pandas as pd
import json
from transformers import TrainingArguments as HfTrainingArguments
from synthtexteval.eval.downstream.classify.train_classifier import TrainingArguments, ModelArguments, Classifier, Arguments
from synthtexteval.utils.utils import create_classification_dataset
from synthtexteval.utils.filtering import process_df

#### Classification: Creating the dataset

Filtering data and creating a structured format out of raw synthetic text.

We have assumed that the synthetic text is generated with labels (the labels typically serve as control codes in most setups).

In [ ]:
# Create mapping from the original data
# And creating a test set for evaluating the model once it is trained
from datasets import load_from_disk, concatenate_datasets
from synthtexteval.utils.utils import encode_labels

In [ ]:
tab_data = load_from_disk('./data/generator/data/tab/')
col_names = [i for i in tab_data['train'].column_names if i not in ['country', 'text', 'year']]
tab_data = tab_data.remove_columns(col_names)
tab_data['train'] = concatenate_datasets([tab_data['train'], tab_data['validation'], tab_data['test']])

_, _ = encode_labels(tab_data['train'], label_column = 'country', json_mapping_exists = False, 
                                json_mapping_path = f'./data/benchmark/classification/data/{dataset_name}-mapping.json', multilabel=False)


In [ ]:
tab_data = load_from_disk('./data/generator/data/tab/')
df, _ = encode_labels(tab_data['validation'], label_column = 'country', json_mapping_exists = True, 
                                json_mapping_path = f'./data/benchmark/classification/data/{dataset_name}-mapping.json', multilabel=False)
test_file_path = f'./data/benchmark/classification/data/test/{dataset_name}/test.csv'
print(f"Saving test file to: {test_file_path}")
df.to_csv(test_file_path)

In [ ]:
# Use this mapping for converting the synthetic data to a structured format
df = pd.read_csv(synthetic_data_path)
df = process_df(df, text_column = 'output_text')
_, _, _ = create_classification_dataset(df, label_column = 'country', json_mapping_path = f'./data/benchmark/classification/data/{dataset_name}-mapping.json', json_mapping_exists = True,
                                        output_dir = f'./data/benchmark/classification/data/{model_config}', multilabel = False, train_ratio = 0.7, test_ratio = 0.15, val_ratio = 0.15)

In [ ]:
with open(f'./data/benchmark/classification/data/{dataset_name}-mapping.json') as f:
    data = json.load(f)
    n_labels_task = len(data)
print(f"Number of labels: {n_labels_task}")

#### Classification: Training the model

This can also be run as a script. Sample script provided in eval.downstream.classify

In [ ]:
if __name__ == "__main__":
        train_args, model_args = TrainingArguments(), ModelArguments()

        model_args.model_name = 'bert-base-uncased'
        model_args.text_field = 'output_text'
        model_args.label_field = 'Label'
        model_args.path_to_dataset = f'./data/benchmark/classification/data/{model_config}'
        model_args.path_to_model = f'./data/benchmark/classification/models/{model_args.model_name}_{model_config}'
        model_args.n_labels = n_labels_task
        model_args.is_train = True
        model_args.problem_type = 'single_label_classification'
        args = Arguments(train=train_args, model=model_args)

        print("Training:\n")
        obj = Classifier(args = args)
        obj.finetune_model()

#### Classification: Testing the model

In [ ]:
if __name__ == "__main__":
        train_args, model_args = TrainingArguments(), ModelArguments()
        model_args.is_train = False
        model_args.is_test = True
        model_args.text_field = 'text'
        model_args.label_field = 'Label'

        model_args.model_name = 'bert-base-uncased'
        model_args.path_to_model = f'./data/benchmark/classification/models/{model_args.model_name}_{model_config}'
        model_args.path_to_dataset = f'./data/benchmark/classification/data/test/{dataset_name}/test.csv'
        model_args.path_to_output_csv = f'./data/benchmark/classification/test-results/{model_args.model_name}_{model_config}_test_outputs.csv'
        model_args.path_to_aggregated_results = './data/benchmark/classification/compiled_benchmark_results.csv'

        model_args.n_labels = n_labels_task
        model_args.problem_type = "single_label_classification"
        model_args.retain_columns = ['country', 'year']

        args = Arguments(train=train_args, model=model_args)
        print("Testing:\n")
        obj = Classifier(args = args)
        obj.test_model()

#### Classification: Fairness auditing of the trained classifier

In [ ]:
from synthtexteval.eval.downstream.classify.visualize import tabulate_results

path_to_test_output = f'./data/benchmark/classification/test-results/{model_args.model_name}_{model_config}_test_outputs.csv'
tabulate_results([path_to_test_output], n_labels = n_labels_task, print_fairness=True, subgroup_type="country", problem_type = "multiclass")

### Generic descriptive analysis of synthetic and reference texts

This section uses the balanced, non-paired text samples prepared above. It is a generic corpus-level comparison: row order is not a child, story, prompt-pair, or clinical comparison.

In [ ]:
import nltk
import pandas as pd
from synthtexteval.eval.descriptive.descriptor import TextDescriptor
from synthtexteval.eval.descriptive.arguments import TextDescriptorArgs
nltk.download('punkt_tab')

In [ ]:
synth_df = comparison_df.rename(columns={"source": "output_text"})[["output_text"]].copy()
real_texts = comparison_df["reference"].tolist()
assert len(synth_df) == len(real_texts) > 0
print(f"Prepared {len(synth_df)} balanced synthetic/reference texts for generic comparison.")

#### Text length and distributional comparisons

In [ ]:
descriptor_args = TextDescriptorArgs(
    pkl_file_path=str(RESULTS_DIR / "entity-output.pkl"),
    plt_file_path=str(RESULTS_DIR / "entity-analysis.png"),
    produce_plot=True,
)
desc_analyze = TextDescriptor(texts=synth_df["output_text"].tolist(), args=descriptor_args, reference_texts=real_texts)

In [ ]:
desc_analyze._compare_to_reference_distribution(metrics = ['text-length', 'jaccard', 'cosine'])

In [ ]:
import pyLDAvis
pyLDAvis.enable_notebook()
tm = desc_analyze._topic_modeling_display(num_topics=3)

In [ ]:
tm

### Upstream privacy example (not configured for ICL-PILOT)

Do not execute this section. The project stories are out of scope for this entity-based privacy example, and no ICL-PILOT privacy claim is produced here. The original TAB paths are retained only to preserve the upstream demo.

#### Privacy: Defining the entities

In [ ]:
import re
import pandas as pd
from datasets import load_from_disk

real_texts = load_from_disk('./data/generator/data/tab')
real_texts = real_texts['train']
synth_df = pd.read_csv(synthetic_data_path)

In [ ]:
entities = []
for i in real_texts['annotations']:
    try:
        for annotator in i:
            for entity in i[annotator]['entity_mentions']:
                if(entity['entity_type'] in ['PERSON', 'DATETIME']):
                    entities.append(entity['span_text'])
    except Exception as e:
        continue

In [ ]:
print(len(entities))

#### Evaluating leakage of entities

In [ ]:
from synthtexteval.eval.privacy.metrics import entity_leakage, search_and_compute_EPO

In [ ]:
total_leakage, privacy_analysis = entity_leakage(synth_df['output_text'].tolist(), entities, 'privacy-leakage.pkl')

In [ ]:
print(f"Percentage of leaked entities: {100*total_leakage:.3f} %")

#### Evaluating span memorization

Doing this only for 15 entities as it is time-intensive

In [ ]:
entities = entities[:15]

In [ ]:
t_df = pd.DataFrame({'text': synth_df['output_text'].tolist()[:10]})


search_and_compute_EPO(synth_file = synth_df, ref_file = t_df, 
                       synth_phrase_file_path = 'synth-outputs.csv', ref_phrase_file_path = 'ref-outputs.csv',
                       entity_patterns = fake_entities, max_window_len = 3,
                       text_field = text_field)

In [ ]:
print(f"Percentage of leaked entity contexts: {100*total_leakage:.3f} %")

### Generic automated text-quality evaluation

The metrics below are toolkit outputs for the balanced corpus-level samples configured above. They are not the project-specific linguistic, developmental, or clinical evaluation.

In [ ]:
from synthtexteval.eval.text_quality.metrics import TextQualityEval
from synthtexteval.eval.text_quality.arguments import Arguments, MauveArgs, LMArgs, FrechetArgs

df = comparison_df.copy()
args_ = Arguments(
    frechet=FrechetArgs(source_text_column="source", ref_text_column="reference"),
    mauve=MauveArgs(source_text_column="source", ref_text_column="reference"),
    perplexity=LMArgs(source_text_column="source", ref_text_column="reference"),
)
qual_estimator = TextQualityEval(args_)

In [ ]:
qual_estimator.calculate_perplexity(df)
qual_estimator.calculate_fid_score(df)
if RUN_MAUVE:
    qual_estimator.calculate_mauve_score(df)

In [ ]:
qual_estimator.print_metrics(qual_estimator.return_results())